# Experiment 3 -- aggregate and plot: complementary multi-source transfer

Reads every per-task CSV written by `run_experiment3.py` into
`Results_simulation/experiment3/raw/` (one file per seed, 5 rows -- one
per method), aggregates the Monte Carlo mean +/- SE of the overall and
pairwise-contrast errors, and reproduces the paper's Figure 4 (Section
5.2, "Performance in multi-source and multi-cluster settings").

Run this after the SLURM array in
`Slurm_Scripts/experiment3_complementary/` has finished (or partially
finished).

This experiment compares target-only, source-1-only, source-2-only,
combined-source, and the adaptive selector (Algorithm 3); `METHOD_ORDER`
below covers all 5 methods. The target+source pooled-subspace estimator
(Algorithm 4) is not part of this comparison.

In [ ]:
import sys, os, glob

# Hardcoded (rather than relative to "..") because the kernel's cwd isn't
# guaranteed to be this notebook's directory -- e.g. VS Code's Jupyter
# extension often starts kernels from the workspace root instead.
# EDIT: set this to the local path of your clone of this repository.
PROJECT_ROOT = "/path/to/Transfer_clustering"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "Numerical_Experiments", "Experiments_Script"))

# The figure below uses matplotlib's text.usetex=True, which shells out to
# `latex`/`dvipng`. If a TeX Live install isn't already on PATH, set
# TEXLIVE_BIN to its bin/ directory (e.g. the output of `dirname $(which latex)`).
TEXLIVE_BIN = None
if TEXLIVE_BIN and os.path.isdir(TEXLIVE_BIN) and TEXLIVE_BIN not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = TEXLIVE_BIN + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment3", "raw")
COMBINED_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment3", "combined")
os.makedirs(COMBINED_DIR, exist_ok=True)

paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(paths)} raw result files")
assert paths, f"No CSVs found in {RAW_DIR} -- has the SLURM array finished any tasks yet?"

df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df.head(10)

In [ ]:
METHOD_ORDER = ["target_only", "source1_only", "source2_only", "combined_sources", "adaptive"]
METHOD_LABELS = {
    "target_only": "Target-only",
    "source1_only": "Source-1-only",
    "source2_only": "Source-2-only",
    "combined_sources": "Combined sources",
    "adaptive": "Adaptive",
}
ERROR_COLS = ["error_overall", "error_12", "error_13", "error_23"]

# Sanity check: how many seeds actually landed per method? If well short
# of N_SEEDS in run_experiment3.py, the array job hasn't finished (or some
# tasks failed -- check Slurm_Scripts/.../Error_Messages).
df.groupby("method")["seed"].nunique().reindex(METHOD_ORDER)

In [ ]:
summary = df.groupby("method")[ERROR_COLS].agg(["mean", lambda s: s.std(ddof=1) / np.sqrt(len(s))])
summary.columns = [f"{col}_{stat}" for col, stat in
                   zip([c for c in ERROR_COLS for _ in range(2)], ["mean", "se"] * len(ERROR_COLS))]
summary = summary.reindex(METHOD_ORDER)
summary.to_csv(os.path.join(COMBINED_DIR, "experiment3_summary.csv"))
summary

## Qualitative pattern table

A checkmark means the method's mean error on that contrast is below
`CHECK_THRESHOLD` (a coin-flip-relative cutoff, not a formal test) --
purely for comparing against the paper's expected pattern table, not a
substitute for the numeric `summary` table above.

In [ ]:
CHECK_THRESHOLD = 0.15

pattern = pd.DataFrame(index=METHOD_ORDER, columns=["(1,2)", "(1,3)", "(2,3)"])
for method in METHOD_ORDER:
    for col, err_col in zip(["(1,2)", "(1,3)", "(2,3)"], ["error_12", "error_13", "error_23"]):
        mean_err = summary.loc[method, f"{err_col}_mean"]
        pattern.loc[method, col] = "\u2713" if mean_err < CHECK_THRESHOLD else "\u2717"
pattern.index = [METHOD_LABELS[m] for m in pattern.index]
pattern

## Figure: error by method and contrast

Grouped bar chart, one group per contrast (overall, (1,2), (1,3), (2,3)),
one bar per method with a fixed color (never reassigned across groups),
SE error bars.

In [ ]:
METHOD_COLORS = {
    "target_only": "#2a78d6",
    "source1_only": "#1baf7a",
    "source2_only": "#eda100",
    "combined_sources": "#008300",
    "adaptive": "#4a3aa7",
}
GROUP_LABELS = [r"$\mathcal{L}_{\mathrm{mult}}$", r"$\mathcal{L}_{12}$", r"$\mathcal{L}_{13}$", r"$\mathcal{L}_{23}$"]

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

n_methods = len(METHOD_ORDER)
x = np.arange(len(ERROR_COLS))
width = 0.8 / n_methods

fig, ax = plt.subplots(figsize=(9, 5))
for i, method in enumerate(METHOD_ORDER):
    means = [summary.loc[method, f"{c}_mean"] for c in ERROR_COLS]
    ses = [summary.loc[method, f"{c}_se"] for c in ERROR_COLS]
    offset = (i - (n_methods - 1) / 2) * width
    ax.bar(x + offset, means, width, yerr=ses, capsize=3,
           label=METHOD_LABELS[method], color=METHOD_COLORS[method])

ax.set_xticks(x)
ax.set_xticklabels(GROUP_LABELS)
ax.set_ylabel("misclustering error")
ax.set_title("Experiment 3: error by method and contrast")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(os.path.join(COMBINED_DIR, "experiment3_bars.pdf"), bbox_inches="tight")
plt.show()

d0 = df[df["method"] == "adaptive"].copy()
d0["D0_used"] = pd.to_numeric(d0["D0_used"], errors="coerce")
d0["D0_used"].agg(["mean", "std", "count"])